# UCI 银行营销：联邦学习数据准备

步骤 S1.P1。原始文件只读；每次 Run All 新建数据与结果目录。字段归属和重叠均为构造条件。

本 Notebook 负责处理与校验，未执行密码学 PSI 或联邦训练。运行结果保留在本地，仓库源文件清除输出。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display
candidates = [Path(os.environ.get("VFL_REPO_ROOT", Path.cwd())).resolve()]
candidates += list(candidates[0].parents)
roots = [p for c in candidates for p in (c, c / "HSBC-VFL-personal-project")
         if (p / "modules/m1_data_selection/components/data_preparation.py").is_file()]
if not roots:
    raise RuntimeError("请设置 VFL_REPO_ROOT 为项目仓库路径")
repo = roots[0]
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from modules.m1_data_selection.components.data_preparation import prepare_context, run_pipeline
config_path = repo / "modules/m1_data_selection/configs/uci_bank_preparation.yaml"
ctx = prepare_context(config_path, repo)
print("VFL_RUN_LOCATOR=" + json.dumps({"result_dir": str(ctx["result_dir"])}))
print(json.dumps({k: ctx["fingerprint"][k] for k in
      ("run_id", "config", "config_sha256", "seeds", "git_sha", "git_dirty", "started_at")}, ensure_ascii=False, indent=2))


以上记录本次配置、种子和代码版本。即使 Git 尚未提交，运行目录也保存代码快照和逐文件哈希，实际执行代码可追溯。

In [ ]:
result = run_pipeline(ctx)
display(result["summary"])

双方使用同一训练、验证和测试记录划分。Alice 持有结果，Bob 只持有自己的特征；每方独立使用训练记录拟合填补与编码。表中的行数只描述该公开代理数据。

In [ ]:
display(result["missingness"])

这里显示清洗后、填补前的缺失情况；保留缺失说明，避免把未知值误当成业务上的否定。训练中位数与类别字典见本次 preprocessing_state.json。

In [ ]:
print(json.dumps(result["checks"], ensure_ascii=False, indent=2))
print(json.dumps(result["overlap"], ensure_ascii=False, indent=2))

raw_unchanged、split_disjoint、party_alignment 和 bob_has_no_outcomes 为处理校验。重叠规模由配置人为构造，只用于之后的记录级协议接入；cryptographic_psi_executed 和 federated_training_executed 为 false。

In [ ]:
print("新数据：", result["data_dir"])
print("运行结果：", result["result_dir"])
print("解释边界：")
for limit in ctx["config"]["limits"]:
    print("-", limit)

本次得到可供下一阶段消费的数据文件和显式字段契约。尚未估计模型增益、营销因果收益或生产性能；不得将处理完成记为 M1 全部验收通过。